# Chapter 9: Multi-Agent Systems

In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 9.1 Why multi-agent?

## 9.2 Three collaboration patterns

### Workflow

**Listing 9.1** Workflow: Sequential execution example

In [ ]:
# Workflow pattern: Developer defines the order
workflow = run_sequential(agents=[
    researcher,  # 1. Research
    coder,       # 2. Write code
    writer,      # 3. Write content
    reviewer     # 4. Review
])

### Agent as Tool

**Listing 9.2** Agent as tool: Agent registration

In [ ]:
# Agent as Tool pattern: Wrap agents as tools
assistant = Agent(
    instructions="Answer user questions",
    tools=[
        AgentTool(researcher),   # Call when research is needed
        AgentTool(calculator),   # Call when calculation is needed
    ]
)

### Transfer

**Listing 9.3** Transfer: LLM-selected agent routing

In [ ]:
# Transfer pattern: LLM selects the next agent
router = Agent(
    name="router",
    instructions="Analyze the request and route to the appropriate expert",
    sub_agents=[code_expert, math_expert, writing_expert]
)

### Combining patterns

## 9.3 Agent workflow: Defining order in code

### 9.3.1 Workflow design principles

**Listing 9.4** Workflow design: Agent inheritance

In [ ]:
class SequentialWorkflow(Agent):
    async def run(self, user_input=None, context=None, verbose=False) -> AgentResult:
        ...

class ParallelWorkflow(Agent):
    async def run(self, user_input=None, context=None, verbose=False) -> AgentResult:
        ...

class LoopWorkflow(Agent):
    async def run(self, user_input=None, context=None, verbose=False) -> AgentResult:
        ...

### 9.3.2 Sequential: Step-by-step execution

**Listing 9.5** SequentialWorkflow implementation

In [ ]:
class SequentialWorkflow(Agent):

    def __init__(self, agents: List[Agent], name: str = "sequential_workflow"):
        self.agents = agents
        self.name = name

    async def run(
        self,
        user_input: Optional[str] = None,
        context: Optional[ExecutionContext] = None,
        verbose: bool = False,
    ) -> AgentResult:
        result = None

        for i, agent in enumerate(self.agents):
            if context is not None:
                # Reset state for next agent execution
                context.final_result = None
                context.current_step = 0

            if i == 0:
                # First agent: pass both user_input and context
                result = await agent.run(user_input, context=context, verbose=verbose)
            else:
                # Subsequent agents: pass only context (no user_input)
                result = await agent.run(context=context, verbose=verbose)

            context = result.context

        return result

### 9.3.3 Parallel: Concurrent execution

**Listing 9.6** ParallelWorkflow implementation and merge

In [ ]:
class ParallelWorkflow(Agent):

    def __init__(self, agents: List[Agent], name: str = "parallel_workflow"):
        self.agents = agents
        self.name = name

    async def run(
        self,
        user_input: Optional[str] = None,
        context: Optional[ExecutionContext] = None,
        verbose: bool = False,
    ) -> AgentResult:
        # Count existing events (to distinguish newly added ones)
        existing_event_count = len(context.events) if context else 0

        # Run all agents concurrently (pass same context)
        results: List[AgentResult] = await asyncio.gather(
            *[agent.run(user_input, context=context, verbose=verbose) 
              for agent in self.agents]
        )

        # Create merged context
        merged_context = ExecutionContext()

        # 1. Copy existing context events
        if context is not None:
            for event in context.events:
                merged_context.add_event(event)

        # 2. Add only new events from each agent (after existing_event_count)
        seen_user_event = (context is not None)

        for result in results:
            new_events = result.context.events[existing_event_count:]

            for event in new_events:
                if event.author == "user":
                    # Add user event only once
                    if not seen_user_event:
                        merged_context.add_event(event)
                        seen_user_event = True
                else:
                    # Add all agent-generated events
                    merged_context.add_event(event)

        # Combine all outputs and return
        combined_output = "\n\n".join(
            f"[{agent.name}]\n{result.output}"
            for agent, result in zip(self.agents, results)
        )

        return AgentResult(output=combined_output, context=merged_context)

### 9.3.4 Loop: Iterative execution

**Listing 9.7** LoopWorkflow implementation and stop condition

In [ ]:
# Stop condition type: (current result, iteration count) -> whether to stop
StopCondition = Callable[[AgentResult, int], bool]

class LoopWorkflow(Agent):

    def __init__(
        self,
        agents: List[Agent],
        stop_condition: Optional[StopCondition] = None,
        max_iterations: int = 10,
        name: str = "loop_workflow",
    ):
        self.agents = agents
        self.stop_condition = stop_condition
        self.max_iterations = max_iterations
        self.name = name

    async def run(
        self,
        user_input: Optional[str] = None,
        context: Optional[ExecutionContext] = None,
        verbose: bool = False,
    ) -> AgentResult:
        result = None
        is_first_agent = True  # Whether this is the first agent overall

        for iteration in range(1, self.max_iterations + 1):
            # One iteration: run all agents sequentially
            for agent in self.agents:
                if context is not None:
                    context.final_result = None
                    context.current_step = 0

                if is_first_agent:
                    # Very first: pass user_input
                    result = await agent.run(user_input, context=context, verbose=verbose)
                    is_first_agent = False
                else:
                    # After: pass only context
                    result = await agent.run(context=context, verbose=verbose)

                context = result.context

            # Check stop condition
            if self.stop_condition and self.stop_condition(result, iteration):
                break

        return result

### 9.3.5 Composing workflows

## 9.4 Agent as Tool: Calling agents as tools

### 9.4.1 Making agents into tools

### 9.4.2 Implementing AgentTool

**Listing 9.8** AgentTool: Agent Wrapper Adapter

In [ ]:
class AgentTool(BaseTool):
    """Adapter that wraps an Agent as a tool."""

    def __init__(
        self,
        agent: Agent,
        input_schema: Optional[Type[BaseModel]] = None,
    ):
        self.agent = agent
        self.input_schema = input_schema

        # Create parameters from input schema
        if input_schema:
            # Extract JSON Schema from Pydantic model
            parameters = input_schema.model_json_schema()
            parameters.pop("$defs", None)
            parameters.pop("title", None)
        else:
            # Default: request string parameter
            parameters = {
                "type": "object",
                "properties": {
                    "request": {
                        "type": "string",
                        "description": "The task or question to delegate"
                    }
                },
                "required": ["request"]
            }

        # Create tool definition from agent's name and description
        tool_def = format_tool_definition(
            agent.name,
            agent.description,
            parameters
        )

        super().__init__(
            name=agent.name,
            description=agent.description,
            tool_definition=tool_def,
        )

    async def execute(self, context: ExecutionContext, **kwargs) -> Any:
        if self.input_schema:
            # Structured input: validate with Pydantic, convert to JSON string
            validated = self.input_schema.model_validate(kwargs)
            request = validated.model_dump_json(exclude_none=True)
        else:
            # Default: use request parameter
            request = kwargs.get("request", str(kwargs))

        # Run agent (in new context)
        result = await self.agent.run(request)

        # Return only the result (intermediate steps not included)
        return result.output

### 9.4.3 Context isolation

### 9.4.4 Practical example: Researcher and coder collaboration

**Listing 9.9** Agent as tool example: Researcher and coder

In [ ]:
from scratch_agents import Agent, LlmClient
from scratch_agents.tools import AgentTool, search_web

llm = LlmClient(model="gpt-5")

# Research specialist: has web search tool
researcher = Agent(
    model=llm,
    name="researcher",
    description="Search the web and summarize findings on any topic",
    instructions="""You are a research specialist.
    Use web_search to find relevant information.
    Summarize your findings concisely.""",
    tools=[search_web],
)

# Coding specialist: has code execution capability
coder = Agent(
    model=llm,
    name="coder",
    description="Write and execute Python code for data analysis and visualization",
    instructions="""You are a Python expert.
    Write clean, working code.
    Execute code to verify results.""",
    code_execution="e2b",
)

## 9.5 Agent transfer pattern

### 9.5.1 Differences between agent as tool and transfer

### 9.5.2 Transfer is also a tool

### 9.5.3 Agent tree

#### Composing the agent tree

**Listing 9.10** Agent class: Transfer attributes

In [ ]:
class Agent:
    def __init__(
        # ... existing parameters ...
        
        # Multi-agent
        sub_agents: Optional[List["Agent"]] = None,
        disallow_transfer_to_peers: bool = False,
    ):
        self.disallow_transfer_to_peers = disallow_transfer_to_peers
        
        # Compose agent tree
        self.sub_agents = sub_agents or []
        self.parent: Optional["Agent"] = None
        
        # Validate and set parent for sub_agents
        self._validate_and_set_sub_agents()

**Listing 9.11** Agent tree validation: Name and parent

In [ ]:
def _validate_and_set_sub_agents(self) -> None:
    """Validate name and parent duplicates in sub_agents and set parent."""
    seen_names = set()
    
    for sub in self.sub_agents:
        # Check for name duplicates
        if sub.name in seen_names:
            raise ValueError(f"Duplicate sub-agent name: '{sub.name}'")
        seen_names.add(sub.name)
        
        # Check for parent duplicates
        if sub.parent is not None:
            raise ValueError(
                f"Agent '{sub.name}' already has parent '{sub.parent.name}'"
            )
        sub.parent = self

#### Determining transfer targets

**Listing 9.12** Transfer targets retrieval

In [ ]:
def _get_transfer_targets(self) -> List["Agent"]:
    """List of targets the current agent can transfer to."""
    targets = []
    
    # 1. Children
    targets.extend(self.sub_agents)
    
    # 2. Parent and siblings
    if self.parent:
        targets.append(self.parent)
        
        # 3. Siblings (optional)
        if not self.disallow_transfer_to_peers:
            for sibling in self.parent.sub_agents:
                if sibling.name != self.name:
                    targets.append(sibling)
    return targets

**Listing 9.13** Agent tree search by name

In [ ]:
def _find_agent(self, name: str) -> Optional["Agent"]:
    """Search by name in the agent tree."""
    # Find root
    root = self
    while root.parent:
        root = root.parent
    
    # Search entire tree from root
    return root._find_in_subtree(name)

def _find_in_subtree(self, name: str) -> Optional["Agent"]:
    """Search in current agent and subtree."""
    if self.name == name:
        return self
    for sub in self.sub_agents:
        if found := sub._find_in_subtree(name):
            return found
    return None

### 9.5.4 Implementing the transfer tool

**Listing 9.14** Transfer tool creation

In [ ]:
def create_transfer_tool(target_agents: List["Agent"]) -> FunctionTool:
    """Create transfer tool from list of transferable agents."""
    
    # Compose agent info
    target_names = [agent.name for agent in target_agents]
    agent_descriptions = []
    for agent in target_agents:
        desc = agent.description or agent.instructions[:100].replace('\n', ' ')
        if len(desc) > 100:
            desc = desc[:100] + "..."
        agent_descriptions.append(f"  - {agent.name}: {desc}")
    
    agent_info = "\n".join(agent_descriptions)
    
    @tool(
        name="transfer_to_agent",
        description=f"""Transfers work to another agent.

Use this tool when the current question belongs to another agent's specialty.

Available agents:
{agent_info}
"""
    )
    def transfer_to_agent(context: ExecutionContext, agent_name: str) -> str:
        """Agent transfer tool."""
        if agent_name not in target_names:
            return f"Error: '{agent_name}' is not valid. Available: {target_names}"
        
        # Only apply first transfer request
        if context.transfer_to is None:
            context.transfer_to = agent_name
            return f"Transferring to {agent_name}..."
        else:
            return f"Transfer already requested to {context.transfer_to}"
    
    # Add enum constraint
    transfer_to_agent.tool_definition["parameters"]["properties"]["agent_name"]["enum"] = target_names
    
    return transfer_to_agent

### 9.5.5 Implementing agent transfer

#### Registering the transfer tool

**Listing 9.15** Registering the transfer tool in run()

In [ ]:
async def run(
    self,
    user_input: Optional[str] = None,
    context: Optional[ExecutionContext] = None,
    verbose: bool = False,
) -> AgentResult:
    
    context = context or ExecutionContext()
    
    # Register transfer tool (if not already)
    if self.name not in context.transfer_tools:
        targets = self._get_transfer_targets()
        if targets:
            from .transfer import create_transfer_tool
            context.transfer_tools[self.name] = create_transfer_tool(targets)
    
    # ... rest of execution logic ...

#### Including the transfer tool in the LLM request

**Listing 9.16** Including transfer tool in LLM request

In [ ]:
async def _prepare_llm_request(self, context: ExecutionContext) -> LlmRequest:
    # Existing tool list
    llm_tools = [t for t in self.tools if t.tool_definition is not None]
    
    # Add transfer tool
    if self.name in context.transfer_tools:
        llm_tools.append(context.transfer_tools[self.name])
    
    # ... create LlmRequest ...

#### Executing the transfer tool

**Listing 9.17** Adding transfer tool to act()

In [ ]:
async def act(
    self,
    context: ExecutionContext,
    tool_calls: List[ToolCall],
) -> List[ToolResult]:
    
    tools_dict = {tool.name: tool for tool in self.tools}
    
    # Include transfer tool
    if self.name in context.transfer_tools:
        transfer_tool = context.transfer_tools[self.name]
        tools_dict[transfer_tool.name] = transfer_tool
    
    # ... tool execution logic ...

#### Detecting and executing transfer

**Listing 9.18** Detecting and executing transfer

In [ ]:
# Execution loop inside run()
while not context.final_result and context.current_step < self.max_steps:
    await self.step(context)
    
    # Check for transfer
    if context.transfer_to:
        next_agent = self._find_agent(context.transfer_to)
        context.transfer_to = None  # Reset flag
        
        if next_agent:
            
            # Delegate (pass only context, no user_input)
            await next_agent.run(None, context, verbose)
            break  # Exit current agent's loop
    
    # Check for final response

## 9.6 A2A: Collaborating across networks

### 9.6.1 The core of A2A: Agent card and task request/response

### 9.6.2 Server: Exposing an agent to the network

### 9.6.3 Client: Calling remote agents

The book illustrates A2A with code samples that are not numbered as listings —
the working implementations live in `scratch_agents/a2a_server.py` (the
`MathAgentExecutor` server example) and `scratch_agents/remote.py` (the
`RemoteAgent` client wrapper).